In [41]:
import os
import rootutils

from tqdm.notebook import tqdm

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import pandas as pd
import re

In [43]:
df_coord_numbs = pd.read_csv("data_cod/coord_numbs.csv")
df_coord_numbs.rename(columns={"smiles": "can_smiles"}, inplace=True)

In [44]:
# your allowed elements
organic = {"C", "H", "O", "N", "S", "F", "Cl", "Br", "P"}

# regex to pull out element symbols (Cl, Br or any capital letter + optional lowercase)
_pattern = re.compile(r'Cl|Br|[A-Z][a-z]?')

def smiles_only_organic(smiles: str) -> bool:
    """Return True if every element token in a SMILES string is in our organic set."""
    tokens = _pattern.findall(smiles)
    return all(tok in organic for tok in tokens)

def col_name_ok(col: str) -> bool:
    """
    Return True if, when splitting the column name on dash/–,
    every part is one of our organic elements.
    """
    parts = re.split(r'[–-]', col)
    return all(part in organic for part in parts)

# 1) Filter rows by can_smiles
df_rows = df_coord_numbs[df_coord_numbs['can_smiles'].apply(smiles_only_organic)]

# 2) Build list of columns to keep (and also keep can_smiles itself)
keep_cols = ['id', 'can_smiles'] + [c for c in df_rows.columns if col_name_ok(c)]

# 3) Slice down to just those columns
df_coord_numbs = df_rows[keep_cols]

#### Leave only organic crystalls:
`C, H, O, N, S, F, Cl, Br, P`

In [45]:
organic_elements = ["C", "H", "O", "N", "S", "F", "Cl", "Br", "P"]

---
## Merging with temperatures:

In [46]:
# Drop rows and columns that contain only zeros:

rows_only_zeros = df_coord_numbs[(df_coord_numbs == 0).all(axis=1)]
columns_only_zeros = df_coord_numbs.loc[:, (df_coord_numbs == 0).all(axis=0)]

df_coord_numbs = df_coord_numbs.drop(columns=columns_only_zeros.columns, index=rows_only_zeros.index)

In [47]:
df_merged_temp = pd.read_csv("data_cod/cod_bradley_merged.csv")
df_merged_temp = df_merged_temp[df_merged_temp["id"].isin(df_coord_numbs["id"])]

# Only bradley, to use part of it for test:
df_bradley_temp = pd.read_csv("data_cod/bradley_with_cif.csv")
df_bradley_temp = df_bradley_temp[df_bradley_temp["id"].isin(df_coord_numbs["id"])]

---
## Creating Train and Test splits:
- 0.2 of Bradley is test, rest is train

In [48]:
TEST_BRADLEY_FRAC = 0.2

In [49]:
df_test = df_bradley_temp.sample(frac=TEST_BRADLEY_FRAC, random_state=42)

df_train = df_merged_temp.drop(index=df_merged_temp[df_merged_temp["id"].isin(df_test["id"])].index)

In [50]:
df_train = pd.merge(df_train, df_coord_numbs, on=["id", "can_smiles"], how="inner")
df_test = pd.merge(df_test, df_coord_numbs, on=["id", "can_smiles"], how="inner")

In [51]:
X_train = df_train.drop(columns=["id", "can_smiles", "T", "cif_path"])
y_train = df_train["T"].astype(float)

X_test = df_test.drop(columns=["id", "can_smiles", "T"])
y_test = df_test["T"].astype(float)

In [78]:
X_test

,Br–Br,C–Br,Br–C,C–C,C–Cl,Cl–C,C–H,H–C,C–N,N–C,...,H–O,O–O,P–H,H–P,P–P,S–H,H–S,S–O,O–S,S–S
0,0.0,0.00,0.0,0.706,0.0,0.0,0.941,0.703,1.294,2.000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0
1,0.0,0.00,0.0,1.548,0.0,0.0,0.720,0.730,0.548,1.750,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0
2,0.0,0.00,0.0,2.526,0.0,0.0,0.519,1.000,0.000,0.000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0
3,0.0,0.00,0.0,1.308,0.0,0.0,0.619,0.610,0.615,1.000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0
4,0.0,0.00,0.0,1.703,0.0,0.0,1.135,0.869,0.236,1.429,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.647,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,0.0,0.50,1.0,1.500,0.0,0.0,1.000,1.000,0.000,0.000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0
390,0.0,0.00,0.0,1.351,0.0,0.0,1.262,0.626,0.237,2.000,...,0.295,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0
391,0.0,0.15,1.0,2.000,0.0,0.0,0.708,1.000,0.146,3.000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0
392,1.0,0.00,0.0,0.000,0.0,0.0,0.000,0.000,0.000,0.000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0


In [77]:
X_train

,Br–Br,C–Br,Br–C,C–C,C–Cl,Cl–C,C–H,H–C,C–N,N–C,...,H–O,O–O,P–H,H–P,P–P,S–H,H–S,S–O,O–S,S–S
0,0.0,0.0,0.0,2.324,0.0,0.0,1.347,0.876,0.000,0.0,...,0.140,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.876,0.0,0.0,0.920,0.957,0.153,3.0,...,0.039,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.000,0.0,...,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,2.000,1.0,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,2.000,1.0,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5188,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.000,0.0,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5189,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.000,0.0,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5190,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.000,0.0,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5191,0.0,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.000,0.0,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Training Catboost:

In [52]:
import optuna
from catboost import CatBoostRegressor
from src.utils import eval_metrics

from sklearn.decomposition import PCA

import numpy as np


In [70]:
# Objective function
def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 300, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 5.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "verbose": 0,
        "task_type": "CPU"
    }

    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)

    y_pred = model.predict(X_test)
    return eval_metrics(y_test, y_pred, "regression")["R2"]

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, n_jobs=8)

# Train final model with best params
best_params = study.best_params
best_params["loss_function"] = "RMSE"
best_params["verbose"] = 0

[I 2025-04-24 22:04:15,433] A new study created in memory with name: no-name-d5d01a5e-e0a9-4a3f-bafa-b667655cc443


[I 2025-04-24 22:04:16,051] Trial 6 finished with value: 0.6766817044445199 and parameters: {'iterations': 526, 'learning_rate': 0.14856894962143957, 'depth': 6, 'l2_leaf_reg': 0.4211680617292804, 'bagging_temperature': 0.9614181566658634, 'random_strength': 1.9332014366678725, 'border_count': 230}. Best is trial 6 with value: 0.6766817044445199.
[I 2025-04-24 22:04:16,411] Trial 5 finished with value: 0.506341422189867 and parameters: {'iterations': 356, 'learning_rate': 0.005852835193889274, 'depth': 5, 'l2_leaf_reg': 0.010547863703789448, 'bagging_temperature': 0.24771133996265526, 'random_strength': 0.00508425155675731, 'border_count': 152}. Best is trial 6 with value: 0.6766817044445199.
[I 2025-04-24 22:04:16,588] Trial 3 finished with value: 0.6767740208629572 and parameters: {'iterations': 880, 'learning_rate': 0.11477098765106826, 'depth': 8, 'l2_leaf_reg': 0.5920993265565944, 'bagging_temperature': 0.1986637711862137, 'random_strength': 0.17053747411420384, 'border_count': 14

In [75]:
best_params["iterations"] = 1000

In [76]:
final_model = CatBoostRegressor(**best_params)
final_model.fit(X_train, y_train, verbose=100)

y_pred = final_model.predict(X_test)
eval_metrics(y_test, y_pred, "regression")


0:	learn: 105.8127004	total: 12.8ms	remaining: 12.8s
100:	learn: 67.2635775	total: 900ms	remaining: 8.01s
200:	learn: 54.7505090	total: 1.71s	remaining: 6.8s
300:	learn: 45.9682266	total: 2.54s	remaining: 5.91s
400:	learn: 40.9813115	total: 3.43s	remaining: 5.12s
500:	learn: 35.4721391	total: 4.25s	remaining: 4.23s
600:	learn: 30.9378458	total: 5.07s	remaining: 3.36s
700:	learn: 27.7884738	total: 5.87s	remaining: 2.5s
800:	learn: 24.8699834	total: 6.68s	remaining: 1.66s
900:	learn: 22.6651007	total: 7.49s	remaining: 823ms
999:	learn: 20.7780652	total: 8.29s	remaining: 0us


{'MSE': 2988.978920531826,
 'RMSE': 54.671554948911286,
 'MAE': 35.228787711968074,
 'R2': 0.6888648958572654}